# Module 08: Custom Module Development

Basilisk's real power comes from extensibility. You can write your own modules in:
- **Python only** — using `Basilisk.utilities.pythonVariableLogger` and custom Python classes
- **C/C++ with Python wrapper** — full performance, SWIG-wrapped like built-in modules

### Learning Objectives
- Understand the BSK module interface (`Reset`, `UpdateState`)
- Create a pure Python FSW module (proportional attitude controller)
- Create a custom Python dynamics observer
- Wire custom modules into the standard BSK pipeline
- Understand the C++ module template structure

---

## 1. BSK Module Interface

Every Basilisk module (C++ or Python) must implement:

```
Reset(CurrentSimNanos)     — called at simulation initialization
UpdateState(CurrentSimNanos) — called every task time step
```

A Python module inherits from `sysModel.SysModel` and uses the messaging system.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('plots', exist_ok=True)

from Basilisk.utilities import SimulationBaseClass, macros, orbitalMotion, unitTestSupport
from Basilisk.utilities import RigidBodyKinematics as rbk
from Basilisk.simulation import spacecraft, gravityEffector, reactionWheelStateEffector
from Basilisk.fswAlgorithms import inertial3D, attTrackingError, rwMotorTorque
from Basilisk.architecture import messaging, bskLogging
from Basilisk.architecture import sysModel

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)
print("Imports OK.")

---

## 2. Custom Python FSW Module: MRP-P Controller

We implement a simple **proportional attitude controller** as a Python module. This replaces `MRP_Feedback` with a transparent pure-Python implementation.

In [ ]:
class MRP_P_Controller(sysModel.SysModel):
    """
    Custom Python FSW module: Proportional MRP attitude controller.

    u = -K * sigma_BR - P * omega_BR

    Inputs:
        guidInMsg   (attGuidMsgPayload)   — attitude guidance error
        vehConfigInMsg (vehicleConfigMsg) — spacecraft inertia

    Outputs:
        cmdTorqueOutMsg (CmdTorqueBodyMsgPayload) — commanded body torque
    """

    def __init__(self):
        super().__init__()
        self.ModelTag = "MRP_P_Controller"   # required identifier

        # ── Tunable parameters ────────────────────────────────────────────────
        self.K = 0.0   # proportional gain (attitude)
        self.P = 0.0   # derivative gain (rate)

        # ── Input messages ────────────────────────────────────────────────────
        self.guidInMsg       = messaging.AttGuidMsg_C()
        self.vehConfigInMsg  = messaging.VehicleConfigMsg_C()

        # ── Output messages ───────────────────────────────────────────────────
        self.cmdTorqueOutMsg = messaging.CmdTorqueBodyMsg()

        # ── Internal state ────────────────────────────────────────────────────
        self._inertia = np.eye(3)   # cached inertia tensor

    def Reset(self, CurrentSimNanos):
        """Called once at simulation start — read static config messages."""
        vehConfig = self.vehConfigInMsg()
        I_flat = vehConfig.ISCPntB_B
        self._inertia = np.array(I_flat).reshape(3, 3)
        print(f"[{self.ModelTag}] Reset: K={self.K:.4f}, P={self.P:.4f}")

    def UpdateState(self, CurrentSimNanos):
        """Called every FSW task time step."""
        # ── Read guidance error message ───────────────────────────────────────
        guidData    = self.guidInMsg()
        sigma_BR    = np.array(guidData.sigma_BR)   # MRP error
        omega_BR_B  = np.array(guidData.omega_BR_B) # rate error in body frame

        # ── PD control law ────────────────────────────────────────────────────
        u = -self.K * sigma_BR - self.P * omega_BR_B

        # ── Write output torque message ───────────────────────────────────────
        torquePayload = messaging.CmdTorqueBodyMsgPayload()
        torquePayload.torqueRequestBody = u.tolist()
        self.cmdTorqueOutMsg.write(torquePayload, CurrentSimNanos, self.moduleID)


# Quick sanity check
ctrl = MRP_P_Controller()
ctrl.K = 0.1
ctrl.P = 0.5
print(f"Custom module created: {ctrl.ModelTag}")

---

## 3. Custom Python Observation Module: Energy Logger

A module that reads spacecraft state and computes/logs mechanical energy — useful for validating integrator quality.

In [ ]:
class OrbitalEnergyMonitor(sysModel.SysModel):
    """
    Custom Python module: monitors orbital specific energy and angular momentum.
    Pure observer — no outputs, logs to internal arrays.
    """

    def __init__(self, mu):
        super().__init__()
        self.ModelTag = "OrbitalEnergyMonitor"
        self.mu = mu   # gravitational parameter

        # Input
        self.scStateInMsg = messaging.SCStateMsg_C()

        # Internal data arrays
        self.times   = []
        self.energy  = []
        self.h_norm  = []

    def Reset(self, CurrentSimNanos):
        self.times  = []
        self.energy = []
        self.h_norm = []

    def UpdateState(self, CurrentSimNanos):
        state = self.scStateInMsg()
        r = np.array(state.r_BN_N)
        v = np.array(state.v_BN_N)

        r_norm = np.linalg.norm(r)
        v_sq   = np.dot(v, v)

        eps = 0.5 * v_sq - self.mu / r_norm           # specific orbital energy
        h   = np.linalg.norm(np.cross(r, v))           # specific angular momentum

        self.times.append(CurrentSimNanos * macros.NANO2SEC)
        self.energy.append(eps)
        self.h_norm.append(h)


print("OrbitalEnergyMonitor module defined.")

---

## 4. Wire Custom Modules into a Full Simulation

In [ ]:
mu_earth = 3.986004418e14
R_earth  = 6371e3

scSim = SimulationBaseClass.SimBaseClass()
dynProcess = scSim.CreateNewProcess("DynamicsProcess", priority=10)
fswProcess = scSim.CreateNewProcess("FswProcess",      priority=5)
dynTask = scSim.CreateNewTask("DynamicsTask", macros.sec2nano(0.5))
fswTask = scSim.CreateNewTask("FswTask",      macros.sec2nano(1.0))
dynProcess.addTask(dynTask)
fswProcess.addTask(fswTask)

# ── Spacecraft ────────────────────────────────────────────────────────────────
scObject = spacecraft.Spacecraft()
scObject.ModelTag = "CustomCtrlSat"
Ixx, Iyy, Izz = 0.025, 0.050, 0.065
I3x3 = np.diag([Ixx, Iyy, Izz])
scObject.hub.mHub = 10.0
scObject.hub.IHubPntBc_B = unitTestSupport.np2EigenMatrix3d(I3x3)
scObject.hub.r_BcB_B = [[0.0], [0.0], [0.0]]

oe = orbitalMotion.ClassicElements()
oe.a = R_earth + 500e3
oe.e = oe.i = oe.Omega = oe.omega = oe.f = 0.0
r0, v0 = orbitalMotion.elem2rv(mu_earth, oe)
scObject.hub.r_CN_NInit = r0.tolist()
scObject.hub.v_CN_NInit = v0.tolist()
scObject.hub.sigma_BNInit   = [[0.3], [-0.2], [0.1]]
scObject.hub.omega_BN_BInit = [[0.01], [0.02], [-0.005]]

gravFactory = gravityEffector.GravBodyDataVector()
earth = gravFactory.createEarth()
earth.isCentralBody = True
scObject.gravField.gravBodies = gravFactory
scSim.AddModelToTask("DynamicsTask", scObject)

# ── RWs ───────────────────────────────────────────────────────────────────────
rwFactory = reactionWheelStateEffector.rwFactory()
varRWModel = reactionWheelStateEffector.BalancedWheels
for axis in [[1,0,0],[0,1,0],[0,0,1]]:
    rwFactory.create('Honeywell_HR16', gsHat_B=axis,
                     maxMomentum=50.0, Omega=0.0, RWModel=varRWModel)
rwStateEffector = reactionWheelStateEffector.ReactionWheelStateEffector()
rwStateEffector.ModelTag = "RWArray"
rwFactory.addToSpacecraft(scObject.ModelTag, rwStateEffector, scObject)
scSim.AddModelToTask("DynamicsTask", rwStateEffector)

# ── FSW: standard guidance + error modules ────────────────────────────────────
inertial3DObj = inertial3D.inertial3D()
inertial3DObj.ModelTag = "inertial3D"
inertial3DObj.sigma_R0N = [0.0, 0.0, 0.0]
scSim.AddModelToTask("FswTask", inertial3DObj)

attErrObj = attTrackingError.attTrackingError()
attErrObj.ModelTag = "attTrackingError"
attErrObj.attNavInMsg.subscribeTo(scObject.attOutMsg)
attErrObj.attRefInMsg.subscribeTo(inertial3DObj.attRefOutMsg)
scSim.AddModelToTask("FswTask", attErrObj)

# ── FSW: CUSTOM Python controller ─────────────────────────────────────────────
myController = MRP_P_Controller()
myController.K = 2.0 * 0.7 * 0.15 * (Ixx + Iyy + Izz)
myController.P = 0.15**2 * (Ixx + Iyy + Izz)

myController.guidInMsg.subscribeTo(attErrObj.attGuidOutMsg)
myController.vehConfigInMsg.subscribeTo(scObject.vehicleConfigOutMsg)

scSim.AddModelToTask("FswTask", myController)

# ── RW allocation — connect from custom controller ────────────────────────────
rwMotorTorqueObj = rwMotorTorque.rwMotorTorque()
rwMotorTorqueObj.ModelTag = "rwMotorTorque"
rwMotorTorqueObj.controlAxes_B = [1,0,0, 0,1,0, 0,0,1]
rwMotorTorqueObj.vehControlInMsg.subscribeTo(myController.cmdTorqueOutMsg)  # custom output!
rwMotorTorqueObj.rwParamsInMsg.subscribeTo(rwFactory.rwParamsMsgs[0])
rwStateEffector.rwMotorCmdInMsg.subscribeTo(rwMotorTorqueObj.rwMotorTorqueOutMsg)
scSim.AddModelToTask("FswTask", rwMotorTorqueObj)

# ── Custom observer module ─────────────────────────────────────────────────────
energyMonitor = OrbitalEnergyMonitor(mu_earth)
energyMonitor.scStateInMsg.subscribeTo(scObject.scStateOutMsg)
scSim.AddModelToTask("DynamicsTask", energyMonitor)

# ── Standard recorders ────────────────────────────────────────────────────────
attErrRec = attErrObj.attGuidOutMsg.recorder()
scSim.AddModelToTask("FswTask", attErrRec)

scSim.InitializeSimulation()
scSim.ConfigureStopTime(macros.sec2nano(300.0))
scSim.ExecuteSimulation()

print("Simulation with custom modules complete.")

In [ ]:
# ── Plot attitude error using custom controller ────────────────────────────────
t = np.array(attErrRec.times()) * macros.NANO2SEC
err = np.degrees(4 * np.arctan(np.linalg.norm(attErrRec.sigma_BR, axis=1)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(t, err, color='steelblue')
axes[0].axhline(1.0, color='red', linestyle='--', label='1 deg')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Attitude Error (deg)')
axes[0].set_title('Custom Python Controller — Attitude Error')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Energy conservation from custom observer
t_e = np.array(energyMonitor.times)
E   = np.array(energyMonitor.energy)
E_err = (E - E[0]) / abs(E[0]) * 100

axes[1].plot(t_e, E_err, color='darkorange')
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Orbital Energy Error (%)')
axes[1].set_title('Custom Observer — Energy Conservation')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/08_custom_module_results.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Custom controller: final attitude error = {err[-1]:.4f} deg")
print(f"Custom observer: max energy error = {abs(E_err).max():.2e} %")

---

## 5. C++ Module Template Structure

For production use, custom modules are implemented in C++ and SWIG-wrapped. Here is the template structure:

In [ ]:
print("""
# ── C++ Module Template (myModule.h) ──────────────────────────────────────────

#pragma once
#include "architecture/_GeneralModuleFiles/sys_model.h"
#include "architecture/msgPayloadDefC/AttGuidMsgPayload.h"
#include "architecture/msgPayloadDefC/CmdTorqueBodyMsgPayload.h"
#include "architecture/messaging/messaging.h"

class MyFswModule : public SysModel {
public:
    MyFswModule();
    ~MyFswModule();

    void Reset(uint64_t CurrentSimNanos) override;
    void UpdateState(uint64_t CurrentSimNanos) override;

    // ── Parameters ─────────────────────────────────────────────────────────
    double K = 0.0;    // proportional gain
    double P = 0.0;    // derivative gain

    // ── Input messages ──────────────────────────────────────────────────────
    ReadFunctor<AttGuidMsgPayload>      guidInMsg;

    // ── Output messages ─────────────────────────────────────────────────────
    Message<CmdTorqueBodyMsgPayload>    cmdTorqueOutMsg;

private:
    // internal state
};


# ── C++ Module Implementation (myModule.cpp) ──────────────────────────────────

#include "myModule.h"

MyFswModule::MyFswModule() = default;
MyFswModule::~MyFswModule() = default;

void MyFswModule::Reset(uint64_t CurrentSimNanos) {
    // one-time initialization
}

void MyFswModule::UpdateState(uint64_t CurrentSimNanos) {
    AttGuidMsgPayload guidData = this->guidInMsg();

    // Compute torque
    Eigen::Vector3d sigma_BR = cArray2EigenVector3d(guidData.sigma_BR);
    Eigen::Vector3d omega_BR = cArray2EigenVector3d(guidData.omega_BR_B);
    Eigen::Vector3d u = -this->K * sigma_BR - this->P * omega_BR;

    CmdTorqueBodyMsgPayload out;
    eigenVector3d2CArray(u, out.torqueRequestBody);
    this->cmdTorqueOutMsg.write(&out, this->moduleID, CurrentSimNanos);
}


# ── SWIG interface file (myModule.i) ──────────────────────────────────────────

%module myModule
%include <myModule.h>


# ── CMakeLists.txt addition ───────────────────────────────────────────────────

# add_basilisk_module(myModule myModule.cpp myModule.i)
""")

---

## 6. Tips for Custom Module Development

| Tip | Details |
|---|---|
| **Start in Python** | Prototype quickly, then port to C++ if performance needed |
| **Use existing message types** | Browse `architecture/msgPayloadDefC/` — 100+ pre-defined types |
| **Unit test every module** | Put tests in `_UnitTest/` — use `pytest` |
| **Never modify BSK source** | Create modules in a separate project directory |
| **Use `bskLogging`** | `BSK_INFORMATION`, `BSK_WARNING`, `BSK_ERROR` for logging |
| **Check message validity** | `if not self.guidInMsg.isLinked(): return` |

---

## Summary

- Python modules inherit from `sysModel.SysModel` and implement `Reset()` + `UpdateState()`
- Input messages use `ReadFunctor` (C++) or `_C()` suffix (Python)
- Output messages use `Message<T>` (C++) or `messaging.XxxMsg()` (Python)
- The custom module output message connects to standard BSK modules identically to built-in modules
- For flight software, C++ modules are preferred for performance; Python modules are perfect for research and prototyping

**Next: [09 - Capstone: Full 6-DOF Mission](09_capstone.ipynb)**